In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np

In [2]:
df_train = pd.read_parquet("/Users/tharmmm/Documents/store_sale_project/train_featured.parquet")
df_test = pd.read_parquet("/Users/tharmmm/Documents/store_sale_project/test_featured.parquet")

In [3]:
df_train.head()

,date,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,is_holiday,...,promo_lag_7,city,state,type,cluster,dcoilwtico,day_of_month,is_month_start,is_month_end,sales
0,2013-01-01,1,AUTOMOTIVE,0,1,1,2013,1,0,1,...,NaN,Quito,Pichincha,D,13,93.14,1,1,0,0.0
1,2013-01-02,1,AUTOMOTIVE,0,2,1,2013,1,0,0,...,NaN,Quito,Pichincha,D,13,93.14,2,0,0,2.0
2,2013-01-03,1,AUTOMOTIVE,0,3,1,2013,1,0,0,...,NaN,Quito,Pichincha,D,13,92.97,3,0,0,3.0
3,2013-01-04,1,AUTOMOTIVE,0,4,1,2013,1,0,0,...,NaN,Quito,Pichincha,D,13,93.12,4,0,0,3.0
4,2013-01-05,1,AUTOMOTIVE,0,5,1,2013,1,1,0,...,NaN,Quito,Pichincha,D,13,93.12,5,0,0,5.0


In [4]:
df_test.head()

,store_nbr,family,onpromotion,day_of_week,month,year,week_of_year,is_weekend,day_of_month,is_month_start,...,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7,city,state,type,cluster,dcoilwtico,transactions,transactions_lag_7
0,1,AUTOMOTIVE,0,2.0,8.0,2017.0,33,0.0,16.0,0.0,...,5.035714,3.287784,NaN,Quito,Pichincha,D,13,46.80,NaN,NaN
1,1,AUTOMOTIVE,0,3.0,8.0,2017.0,33,0.0,17.0,0.0,...,NaN,NaN,NaN,Quito,Pichincha,D,13,47.07,NaN,NaN
2,1,AUTOMOTIVE,0,4.0,8.0,2017.0,33,0.0,18.0,0.0,...,NaN,NaN,NaN,Quito,Pichincha,D,13,48.59,NaN,NaN
3,1,AUTOMOTIVE,0,5.0,8.0,2017.0,33,1.0,19.0,0.0,...,NaN,NaN,NaN,Quito,Pichincha,D,13,48.59,NaN,NaN
4,1,AUTOMOTIVE,0,6.0,8.0,2017.0,33,1.0,20.0,0.0,...,NaN,NaN,NaN,Quito,Pichincha,D,13,48.59,NaN,NaN


In [7]:
len(df_train['date'].unique())

1684

In [8]:
print(df_train["date"].min(), "to", df_train["date"].max())

2013-01-01 00:00:00 to 2017-08-15 00:00:00


Split train and validation set

Train: 2013-01-01 to 2017-07-31 (~4.5 years)
Validation: 2017-08-01 to 2017-08-15 (15 days)
Test: 2017-08-16 to 2017-08-31 (16 days)

In [ ]:
val_start = "2017-08-01"

X_train = df_train[df_train["date"] < val_start].drop(columns=["sales", "date"])
y_train = df_train[df_train["date"] < val_start]["sales"]

X_val = df_train[df_train["date"] >= val_start].drop(columns=["sales", "date"])
y_val = df_train[df_train["date"] >= val_start]["sales"]


In [ ]:
for col in ["family", "city", "state", "type"]:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

Model

In [9]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    enable_categorical=True
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50,
    verbose=100
)


NameError: name 'lgb' is not defined

In [ ]:
# Evaluate both
lgb_pred = lgb_model.predict(X_val)
xgb_pred = xgb_model.predict(X_val)

print(f"LightGBM RMSE: {np.sqrt(mean_squared_error(y_val, lgb_pred)):.4f}")
print(f"XGBoost RMSE: {np.sqrt(mean_squared_error(y_val, xgb_pred)):.4f}")